# Time-Frequency Analysis — Chapter 3
# 時間周波数解析 — 第3章

## The Wigner–Ville Distribution
## Wigner–Ville 分布

---
> **Reference:** Cohen, *Time-Frequency Analysis*, Ch. 4–5; Wigner (1932); Ville (1948).

## 3.1 Definition
## 3.1 定義

**English.**  
The **Wigner–Ville Distribution (WVD)** is defined as:

$$\boxed{W_x(t, f) = \int_{-\infty}^{\infty} x\!\left(t + \frac{\tau}{2}\right) x^*\!\left(t - \frac{\tau}{2}\right) e^{-j2\pi f\tau}\, d\tau}$$

It was introduced by Wigner (1932) in quantum mechanics as a *quasi-probability distribution*, and adapted for signal analysis by Ville (1948).  

The WVD is the **Fourier transform of the instantaneous autocorrelation**:

$$R_x(t, \tau) = x\!\left(t + \frac{\tau}{2}\right) x^*\!\left(t - \frac{\tau}{2}\right) \xrightarrow{\mathcal{F}_\tau} W_x(t, f)$$

For numerical computation on an analytic signal $z(t) = x(t) + j\hat{x}(t)$  
(where $\hat{x}$ is the Hilbert transform), the WVD avoids aliasing and cross-component interference from negative frequencies.

---

**日本語.**  
**Wigner–Ville 分布（WVD）** は以下のように定義されます：

$$\boxed{W_x(t, f) = \int_{-\infty}^{\infty} x\!\left(t + \frac{\tau}{2}\right) x^*\!\left(t - \frac{\tau}{2}\right) e^{-j2\pi f\tau}\, d\tau}$$

Wigner（1932年）が量子力学における*準確率分布*として導入し、Ville（1948年）が信号解析に応用しました。

WVD は**瞬時自己相関の Fourier 変換**です：

$$R_x(t, \tau) = x\!\left(t + \frac{\tau}{2}\right) x^*\!\left(t - \frac{\tau}{2}\right) \xrightarrow{\mathcal{F}_\tau} W_x(t, f)$$

## 3.2 Mathematical Properties
## 3.2 数学的性質

**English.**  

### Real-valued
$$W_x(t, f) \in \mathbb{R} \quad \forall\, t, f$$
Proof: $W_x^* = \int x^*(t+\tau/2)x(t-\tau/2)e^{+j2\pi f\tau}d\tau = W_x$ (by substitution $\tau \to -\tau$).

### Time marginal
$$\int_{-\infty}^{\infty} W_x(t, f)\, df = |x(t)|^2$$

### Frequency marginal
$$\int_{-\infty}^{\infty} W_x(t, f)\, dt = |X(f)|^2$$

### Total energy (Moyal's formula)
$$\iint W_x(t, f)\, dt\, df = \int |x(t)|^2\, dt = E$$

### Time-shift covariance
$$x(t) \to x(t - t_0) \implies W_x(t, f) \to W_x(t - t_0, f)$$

### Frequency-shift covariance
$$x(t) \to x(t)e^{j2\pi f_0 t} \implies W_x(t, f) \to W_x(t, f - f_0)$$

### Instantaneous frequency
$$\bar{f}(t) = \frac{\int f\, W_x(t, f)\, df}{\int W_x(t, f)\, df} = \frac{1}{2\pi}\frac{d}{dt}\arg x(t)$$

---

**日本語.**  

### 実数値性
$W_x(t, f)$ は常に実数値を取ります（$\tau \to -\tau$ の置換から証明）。

### 時間マージナル
$$\int_{-\infty}^{\infty} W_x(t, f)\, df = |x(t)|^2$$

### 周波数マージナル  
$$\int_{-\infty}^{\infty} W_x(t, f)\, dt = |X(f)|^2$$

### Moyalの式（全エネルギー）
$$\iint W_x(t, f)\, dt\, df = E$$

### 瞬時周波数
$$\bar{f}(t) = \frac{\int f\, W_x(t, f)\, df}{\int W_x(t, f)\, df} = \frac{1}{2\pi}\frac{d}{dt}\arg x(t)$$

## 3.3 Cross-Terms: The Fundamental Problem
## 3.3 クロスターム：根本的な問題

**English.**  
For a **multi-component signal** $x(t) = x_1(t) + x_2(t)$:

$$W_x(t, f) = \underbrace{W_{x_1}(t, f) + W_{x_2}(t, f)}_{\text{auto-terms 自己項}} + \underbrace{2\,\text{Re}\left[W_{x_1 x_2}(t, f)\right]}_{\text{cross-term クロスターム}}$$

where the **cross-term** (interference term) is:

$$W_{x_1 x_2}(t, f) = \int x_1\!\left(t + \frac{\tau}{2}\right) x_2^*\!\left(t - \frac{\tau}{2}\right) e^{-j2\pi f\tau}\, d\tau$$

Cross-terms appear **midway between** the auto-terms in the time-frequency plane and can be larger than the auto-terms in magnitude. They are **oscillatory** (they carry a phase) — a key property exploited by kernel design (Cohen's class).

For two Gaussian atoms centred at $(t_1, f_1)$ and $(t_2, f_2)$, the cross-term is centred at
$\left(\frac{t_1+t_2}{2}, \frac{f_1+f_2}{2}\right)$ and oscillates at rate $f_1 - f_2$.

---

**日本語.**  
**多成分信号** $x(t) = x_1(t) + x_2(t)$ に対して：

$$W_x = \underbrace{W_{x_1} + W_{x_2}}_{\text{自己項}} + \underbrace{2\,\text{Re}\left[W_{x_1 x_2}\right]}_{\text{クロスターム}}$$

クロスタームは時間周波数平面上で**自己項の中間点**に現れ、振動的（oscillatory）な性質を持ちます。これはCohenのクラスにおけるカーネル設計で利用される重要な性質です。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import hilbert

plt.rcParams.update({'font.size': 11, 'figure.dpi': 120})

In [ ]:
# ─── Wigner-Ville Distribution (discrete implementation) ──────────────────────
# Wigner-Ville 分布（離散実装）

def wvd(x, N_fft=None):
    """
    Compute the Wigner-Ville Distribution of the analytic signal z.
    Input x should be the analytic signal (complex).
    Returns W[time, freq] of shape (len(x), N_fft//2).
    アナリティック信号 z の WVD を計算する。
    """
    N = len(x)
    if N_fft is None:
        N_fft = 2 * N
    W = np.zeros((N, N_fft // 2), dtype=float)

    for n in range(N):
        # Maximum lag without going out of bounds
        tau_max = min(n, N - 1 - n, N_fft // 2 - 1)
        lags = np.arange(-tau_max, tau_max + 1)

        # Instantaneous autocorrelation
        R = np.zeros(N_fft, dtype=complex)
        R[lags % N_fft] = (
            x[n + lags] * np.conj(x[n - lags])
        )
        # Fourier transform → WVD slice at time n
        wvd_slice = np.real(np.fft.fft(R))
        W[n, :] = wvd_slice[:N_fft // 2]

    return W

print("WVD function defined / WVD関数を定義しました")

In [ ]:
# ─── WVD of a linear chirp signal ─────────────────────────────────────────────
# 線形チャープ信号の WVD

fs = 256
T  = 1.0
t  = np.linspace(0, T, int(fs * T), endpoint=False)

# Linear chirp: 10 Hz → 100 Hz
f0, f1 = 10.0, 100.0
chirp_rate = (f1 - f0) / T
x_real = np.cos(2 * np.pi * (f0 * t + 0.5 * chirp_rate * t**2))

# Analytic signal (one-sided spectrum, avoids cross-terms from neg. freqs)
# アナリティック信号（負周波数からのクロスタームを回避）
z = hilbert(x_real)

N_fft = 512
W = wvd(z, N_fft=N_fft)
freqs = np.linspace(0, fs / 2, N_fft // 2)

# True instantaneous frequency: f(t) = f0 + chirp_rate * t
f_inst = f0 + chirp_rate * t

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].pcolormesh(t, freqs, W.T, cmap='RdBu_r',
                   vmin=-W.max()*0.8, vmax=W.max()*0.8, shading='gouraud')
axes[0].plot(t, f_inst, 'k--', lw=1.5, label='True IF')
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Frequency (Hz)')
axes[0].set_ylim(0, fs / 2)
axes[0].set_title('WVD of linear chirp\nWVD（線形チャープ）')
axes[0].legend()

# Marginal check: time marginal ≈ |x(t)|²
df = freqs[1] - freqs[0]
time_marginal = W.sum(axis=1) * df
axes[1].plot(t, np.abs(z)**2, 'b-', lw=2, label=r'$|z(t)|^2$')
axes[1].plot(t, time_marginal, 'r--', lw=1.5, label='WVD time marginal')
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Power')
axes[1].set_title('Time marginal check\n時間マージナルの検証')
axes[1].legend()

plt.tight_layout()
plt.savefig('../figures/03_wvd_chirp.png', bbox_inches='tight')
plt.show()

In [ ]:
# ─── Cross-terms: two-component signal ────────────────────────────────────────
# クロスターム：2成分信号

fs = 256
t  = np.linspace(0, 1.0, fs, endpoint=False)

# Two Gaussian atoms at different TF locations
# 異なる時間周波数位置にある2つのガウスアトム
sigma = 0.08
a1 = np.exp(-((t - 0.25)**2) / (2*sigma**2)) * np.exp(1j * 2*np.pi * 40 * t)
a2 = np.exp(-((t - 0.75)**2) / (2*sigma**2)) * np.exp(1j * 2*np.pi * 80 * t)

z_combined = a1 + a2

N_fft = 512
W1  = wvd(a1, N_fft=N_fft)
W2  = wvd(a2, N_fft=N_fft)
W12 = wvd(z_combined, N_fft=N_fft)
freqs = np.linspace(0, fs / 2, N_fft // 2)

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
vmax = max(np.abs(W12).max(), np.abs(W1).max(), np.abs(W2).max()) * 0.8
kw = dict(cmap='RdBu_r', vmin=-vmax, vmax=vmax, shading='gouraud')

axes[0].pcolormesh(t, freqs, W1.T + W2.T, **kw)
axes[0].set_title('Auto-terms only (W₁ + W₂)\n自己項のみ')

axes[1].pcolormesh(t, freqs, W12.T, **kw)
axes[1].set_title('WVD of combined signal\n合成信号の WVD')
# Annotate cross-term location
axes[1].annotate('Cross-term\nクロスターム', xy=(0.5, 60), xytext=(0.5, 100),
                 arrowprops=dict(arrowstyle='->', color='black'), fontsize=9,
                 ha='center')

# Cross-term alone
cross = W12 - W1 - W2
axes[2].pcolormesh(t, freqs, cross.T, **kw)
axes[2].set_title('Cross-term alone\nクロスタームのみ')

for ax in axes:
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Frequency (Hz)')
    ax.set_ylim(0, 130)

plt.suptitle('WVD cross-term demonstration\nWVD クロスタームのデモンストレーション',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('../figures/03_wvd_crossterms.png', bbox_inches='tight')
plt.show()

## 3.4 Summary
## 3.4 まとめ

| Property | WVD |
|----------|-----|
| Real-valued | ✅ |
| Non-negative everywhere | ❌ (can be negative) |
| Exact time marginal | ✅ $\int W\,df = |x|^2$ |
| Exact frequency marginal | ✅ $\int W\,dt = |X|^2$ |
| Cross-terms | ❌ Severe for multi-component signals |
| Resolution | Optimal (Heisenberg bound achieved for chirps) |

**English.**  
The WVD achieves **optimal time-frequency resolution** — it perfectly localises linear chirps onto their instantaneous frequency law.  
The fundamental cost is the **cross-term interference** between signal components, which can completely obscure the auto-terms.  
Cohen's class (Chapter 4) provides a systematic framework for trading off resolution against cross-term suppression.

---

**日本語.**  
WVD は**最適な時間周波数分解能**を達成します — 線形チャープをその瞬時周波数則上に完全に局在化します。  
根本的なコストは信号成分間の**クロスターム干渉**であり、自己項を完全に隠してしまうことがあります。  
第4章のCohenのクラスは、分解能とクロスターム抑制のトレードオフに対して体系的なフレームワークを提供します。